## Vector Stroes
A specilzed DB for storing Embeddings

https://docs.langchain.com/oss/python/integrations/vectorstores

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

API_KEY = os.getenv('AZURE_OPENAI_API_KEY')
BASE_URL = os.getenv('OPENAI_BASE_URL')

os.environ['LANGCHAIN_API_KEY'] = os.getenv('LANGSMITH_API_KEY')
os.environ["LANGCHAIN_TRACING_V2"] = "true" 
os.environ['LANGSMITH_PROJECT'] = 'AgenticAITraining' 

In [2]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
embeddings_model=OpenAIEmbeddings(api_key=API_KEY, base_url=BASE_URL, model="text-embedding-3-small")

In [3]:
loader = TextLoader('../data/FinAI.txt')
document = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size = 300, chunk_overlap=50)
docs = text_splitter.split_documents(document)

len(docs)

17

In [4]:
from langchain_chroma import Chroma

vector_store = Chroma(
    collection_name='FinAI',
    embedding_function=embeddings_model
)

In [5]:
from uuid import uuid4

uuids = [str(x+1) for x in range(len(docs))]
print(uuids)

['1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17']


In [6]:
vector_store.add_documents(documents=docs, ids=uuids)

['1',
 '2',
 '3',
 '4',
 '5',
 '6',
 '7',
 '8',
 '9',
 '10',
 '11',
 '12',
 '13',
 '14',
 '15',
 '16',
 '17']

### Similarity Search
Performing a simple similarity search can be done

In [7]:
query = 'Identify speeches that frame war as a defense of political freedom and small nations.'

results = vector_store.similarity_search(query, k=5)

for res in results:
    print(f'* {res.page_content} - [{res.metadata}]') 

* for the right of those who submit to authority to have a voice in their own governments, for the rights and liberties of small nations, for a universal dominion of right by such a concert of free peoples as shall bring peace and safety to all nations and make the world itself at last free. - [{'source': '../data/FinAI.txt'}]
* The world must be made safe for democracy. Its peace must be planted upon the tested foundations of political liberty. We have no selfish ends to serve. We desire no conquest, no dominion. We seek no indemnities for ourselves, no material compensation for the sacrifices we shall freely make. We are - [{'source': '../data/FinAI.txt'}]
* war, into the most terrible and disastrous of all wars, civilization itself seeming to be in the balance. But the right is more precious than peace, and we shall fight for the things which we have always carried nearest our heartsâ€”for democracy, for the right of those who submit to authority to - [{'source': '../data/FinAI.txt'

In [36]:
query = 'Find political speeches about universal peace through a concert of free peoples.'

results = vector_store.similarity_search_with_score(query, k=5 )

for res, score in results:
    print(f' [{score:.3f}]  {res.page_content} - {res.metadata}')

 [0.921]  for the right of those who submit to authority to have a voice in their own governments, for the rights and liberties of small nations, for a universal dominion of right by such a concert of free peoples as shall bring peace and safety to all nations and make the world itself at last free. - {'year': 2025, 'source': '../data/FinAI.txt', 'category': 'finance'}
 [0.966]  The world must be made safe for democracy. Its peace must be planted upon the tested foundations of political liberty. We have no selfish ends to serve. We desire no conquest, no dominion. We seek no indemnities for ourselves, no material compensation for the sacrifices we shall freely make. We are - {'year': 2025, 'source': '../data/FinAI.txt', 'category': 'finance'}
 [1.113]  for the sacrifices we shall freely make. We are but one of the champions of the rights of mankind. We shall be satisfied when those rights have been made as secure as the faith and the freedom of nations can make them. - {'source': '../d

### As Reteriver

We can also cnvert the vectorstore into retriever class. This allows us to easily use it in other LangChain method

In [9]:
query = 'Find political speeches about universal peace through a concert of free peoples.'

In [12]:
reteriever = vector_store.as_retriever(
    search_type="similarity", 
    search_kwargs={"k": 5}
)

results = reteriever.invoke(query)
results

[Document(id='15', metadata={'source': '../data/FinAI.txt'}, page_content='for the right of those who submit to authority to have a voice in their own governments, for the rights and liberties of small nations, for a universal dominion of right by such a concert of free peoples as shall bring peace and safety to all nations and make the world itself at last free.'),
 Document(id='1', metadata={'source': '../data/FinAI.txt'}, page_content='The world must be made safe for democracy. Its peace must be planted upon the tested foundations of political liberty. We have no selfish ends to serve. We desire no conquest, no dominion. We seek no indemnities for ourselves, no material compensation for the sacrifices we shall freely make. We are'),
 Document(id='2', metadata={'source': '../data/FinAI.txt'}, page_content='for the sacrifices we shall freely make. We are but one of the champions of the rights of mankind. We shall be satisfied when those rights have been made as secure as the faith and

In [20]:
print(results[0].model_dump().keys())
print(results[0].page_content)

dict_keys(['id', 'metadata', 'page_content', 'type'])
for the right of those who submit to authority to have a voice in their own governments, for the rights and liberties of small nations, for a universal dominion of right by such a concert of free peoples as shall bring peace and safety to all nations and make the world itself at last free.


### MMR - Maximal marginal Relevence
- Reduce the redundency in the docs
- Relevent but diverse information
    - Fist select the most releven and then diversed

In [ ]:
reteriever = vector_store.as_retriever(
    search_type="mmr", 
    search_kwargs={"k": 2, "fetch_k": 7}
)

results = reteriever.invoke(query)
results

[Document(id='15', metadata={'source': '../data/FinAI.txt'}, page_content='for the right of those who submit to authority to have a voice in their own governments, for the rights and liberties of small nations, for a universal dominion of right by such a concert of free peoples as shall bring peace and safety to all nations and make the world itself at last free.'),
 Document(id='3', metadata={'source': '../data/FinAI.txt'}, page_content='Just because we fight without rancor and without selfish object, seeking nothing for ourselves but what we shall wish to share with all free peoples, we shall, I feel confident, conduct our operations as belligerents without passion and ourselves observe with proud punctilio the principles of right')]

### Running it locally

In [23]:
query = 'Which American president spoke about protecting the liberties of small nations during WWI?'

In [24]:
reteriever_local = vector_store.as_retriever(
    search_type = 'similarity',
    search_kwargs = {'k':5}
)

In [25]:
results = reteriever_local.invoke(query)
results

[Document(id='15', metadata={'source': '../data/FinAI.txt'}, page_content='for the right of those who submit to authority to have a voice in their own governments, for the rights and liberties of small nations, for a universal dominion of right by such a concert of free peoples as shall bring peace and safety to all nations and make the world itself at last free.'),
 Document(id='1', metadata={'source': '../data/FinAI.txt'}, page_content='The world must be made safe for democracy. Its peace must be planted upon the tested foundations of political liberty. We have no selfish ends to serve. We desire no conquest, no dominion. We seek no indemnities for ourselves, no material compensation for the sacrifices we shall freely make. We are'),
 Document(id='2', metadata={'source': '../data/FinAI.txt'}, page_content='for the sacrifices we shall freely make. We are but one of the champions of the rights of mankind. We shall be satisfied when those rights have been made as secure as the faith and

In [34]:
all_ids = vector_store.get()['ids']

vector_store._collection.update(
    ids = all_ids,
    metadatas=[{
        "category": "finance",
        "year": 2025
    } for _ in all_ids]
)

In [35]:
results = reteriever_local.invoke(query)
results

[Document(id='15', metadata={'year': 2025, 'source': '../data/FinAI.txt', 'category': 'finance'}, page_content='for the right of those who submit to authority to have a voice in their own governments, for the rights and liberties of small nations, for a universal dominion of right by such a concert of free peoples as shall bring peace and safety to all nations and make the world itself at last free.'),
 Document(id='1', metadata={'year': 2025, 'source': '../data/FinAI.txt', 'category': 'finance'}, page_content='The world must be made safe for democracy. Its peace must be planted upon the tested foundations of political liberty. We have no selfish ends to serve. We desire no conquest, no dominion. We seek no indemnities for ourselves, no material compensation for the sacrifices we shall freely make. We are'),
 Document(id='2', metadata={'source': '../data/FinAI.txt', 'year': 2025, 'category': 'finance'}, page_content='for the sacrifices we shall freely make. We are but one of the champi